In [ ]:
import os
import numpy as np
import pandas as pd

import scanpy as sc

from essential.steady_state import SteadyStateEstimator

In [ ]:
adata = sc.read_h5ad("/workspace/data/250516_TF_perturbseq/250516_TF_perturbseq.annotated.h5ad")
adata.X = adata.layers["counts"].copy()
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

In [ ]:
embedding_dir = "/workspace/data/e_coli_llm_embeddings"
llm_embeddings = np.load(os.path.join(embedding_dir, "llm_embeddings.npz"))
gene_order = pd.Series(np.arange(len(llm_embeddings["genes"])), index=llm_embeddings["genes"])

In [ ]:
valid_genes = np.intersect1d(adata.var_names, llm_embeddings["genes"])
adata_llm = adata[:, valid_genes].copy()
index_order = gene_order.loc[valid_genes].values
gene_embeddings = llm_embeddings["embeddings"][index_order]
d_embedding = gene_embeddings.shape[1]

In [ ]:
model = SteadyStateEstimator(
    adata_llm,
    expression_type="none",
    model_kwargs={"embeddings": gene_embeddings, "embedding_dim": d_embedding},
    model_class="hardsigmoid2_embedding_steady_state",
)
model.fit(learning_rate=1e-2, n_epochs=1, log_every_n_steps=10, batch_size=100, batch_size_eval=5)